# Medical Knowledge RAG Assistant with Unsloth Dynamic 4-bit Quantization

## Month 2 – Task 3 | Generative AI Internship

This project implements a Retrieval-Augmented Generation (RAG) pipeline using an Unsloth dynamic 4-bit quantized language model. The system retrieves relevant information from domain-specific medical documents and uses the retrieved context to generate grounded answers while keeping GPU memory usage efficient.

In [1]:
!nvidia-smi

Mon Sep 14 17:09:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Environment Setup

The required libraries are installed for loading the Unsloth dynamic 4-bit language model, creating document embeddings, and building the retrieval index used by the RAG pipeline.

In [2]:
!pip install -q -U unsloth
!pip install -q sentence-transformers faiss-cpu pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.

In [3]:
# Verify that the required Task 3 libraries were installed correctly

from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
import faiss
from pypdf import PdfReader

print("✅ Unsloth imported successfully")
print("✅ Sentence Transformers imported successfully")
print("✅ FAISS imported successfully")
print("✅ PyPDF imported successfully")
print("\nEnvironment setup completed successfully.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ Unsloth imported successfully
✅ Sentence Transformers imported successfully
✅ FAISS imported successfully
✅ PyPDF imported successfully

Environment setup completed successfully.


## 3. Loading the Unsloth Dynamic 4-bit Model

A Llama 3.2 3B Instruct model with Unsloth Dynamic 4-bit quantization is loaded for memory-efficient inference. Dynamic quantization selectively preserves higher precision for sensitive model components while keeping most parameters in 4-bit format, reducing GPU memory requirements for the RAG pipeline.

In [4]:
from unsloth import FastLanguageModel
import torch

# Model configuration
max_seq_length = 2048
dtype = None
load_in_4bit = True

model_name = "unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit"

# Load the Dynamic 4-bit model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Enable optimized inference
FastLanguageModel.for_inference(model)

print("✅ Dynamic 4-bit model loaded successfully")
print("Model:", model_name)
print("Maximum sequence length:", max_seq_length)
print("GPU:", torch.cuda.get_device_name(0))

==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ Dynamic 4-bit model loaded successfully
Model: unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit
Maximum sequence length: 2048
GPU: Tesla T4


## 4. GPU Memory Usage

GPU memory usage is measured after loading the Dynamic 4-bit model. This helps demonstrate the memory efficiency of the quantized model and verifies that it can operate within the limited VRAM available on a Google Colab T4 GPU.

In [5]:
# Measure GPU memory usage after loading the Dynamic 4-bit model

allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
reserved_gb = torch.cuda.memory_reserved() / (1024 ** 3)

total_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
free_gb = total_gb - allocated_gb

print("=== GPU Memory Usage ===")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total GPU memory: {total_gb:.2f} GB")
print(f"Memory allocated: {allocated_gb:.2f} GB")
print(f"Memory reserved: {reserved_gb:.2f} GB")
print(f"Approx. free memory: {free_gb:.2f} GB")

=== GPU Memory Usage ===
GPU: Tesla T4
Total GPU memory: 14.56 GB
Memory allocated: 2.24 GB
Memory reserved: 2.27 GB
Approx. free memory: 12.33 GB


## 5. Domain-Specific Medical Documents

To build the retrieval knowledge base, authoritative medical documents from the World Health Organization (WHO) are used. The documents cover hypertension and diabetes, providing domain-specific information that can later be retrieved as context for grounded question answering.

In [6]:
import os
import requests

# Create a folder for the medical source documents
docs_folder = "/content/medical_docs"
os.makedirs(docs_folder, exist_ok=True)

medical_documents = {
    "WHO_Pakistan_Hypertension_Fact_Sheet.pdf":
        "https://cdn.who.int/media/docs/default-source/country-profiles/hypertension/pak_en.pdf?download=true",

    "WHO_Diabetes_Fact_Sheet.pdf":
        "https://cdn.who.int/media/docs/default-source/searo/nde/sde-diabetes-fs.pdf",
}

# Download the documents
for filename, url in medical_documents.items():
    file_path = os.path.join(docs_folder, filename)

    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with open(file_path, "wb") as file:
        file.write(response.content)

    print(f"✅ Downloaded: {filename}")

print("\nMedical document collection prepared successfully.")

✅ Downloaded: WHO_Pakistan_Hypertension_Fact_Sheet.pdf
✅ Downloaded: WHO_Diabetes_Fact_Sheet.pdf

Medical document collection prepared successfully.


## 6. PDF Text Extraction and Inspection

The WHO PDF documents are processed page by page using PyPDF. Extracted text is stored together with its source filename and page number so that retrieved information can later be traced back to the original medical document.

In [7]:
from pypdf import PdfReader
import os
import re

medical_pages = []

# Extract text page by page
for filename in os.listdir(docs_folder):
    if filename.endswith(".pdf"):
        file_path = os.path.join(docs_folder, filename)
        reader = PdfReader(file_path)

        for page_number, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""

            # Basic text cleaning
            text = re.sub(r"\s+", " ", text).strip()

            if text:
                medical_pages.append({
                    "source": filename,
                    "page": page_number,
                    "text": text
                })

        print(f"✅ Processed: {filename} | Pages: {len(reader.pages)}")

print("\n=== Extraction Summary ===")
print("Usable pages extracted:", len(medical_pages))
print(
    "Total extracted characters:",
    sum(len(item["text"]) for item in medical_pages)
)

# Preview the first extracted page
print("\n=== Sample Extracted Text ===")
print("Source:", medical_pages[0]["source"])
print("Page:", medical_pages[0]["page"])
print(medical_pages[0]["text"][:1200])

✅ Processed: WHO_Diabetes_Fact_Sheet.pdf | Pages: 2
✅ Processed: WHO_Pakistan_Hypertension_Fact_Sheet.pdf | Pages: 2

=== Extraction Summary ===
Usable pages extracted: 4
Total extracted characters: 11411

=== Sample Extracted Text ===
Source: WHO_Diabetes_Fact_Sheet.pdf
Page: 1
Quick facts Globally, an estimated 346 million people have diabetes. Three out of four people with diabetes live in low- and • middle-income countries. In the South-East Asia (SEA) Region, nearly 71 million were estimated to be living with diabetes in 2010 and an • equal number had impaired glucose tolerance. Nearly 3.4 million people globally and 1 million in SEA Region die from consequences of high blood sugar every • year. Diabetes exacerbates major infectious diseases such as tuberculosis (TB), malaria and HIV/AIDS. People with • diabetes are three times more likely to develop TB when infected and approximately 15% of TB globally is thought to be due to diabetes. What is diabetes? Diabetes is a chronic cond

## 7. Text Chunking for Retrieval

The extracted medical text is divided into smaller overlapping chunks before embedding. Overlap helps preserve context across chunk boundaries, while source filename and page metadata are retained so retrieved information can be traced back to the original WHO document.

In [8]:
# Split extracted medical text into overlapping chunks

def chunk_text(text, chunk_size=180, overlap=40):
    words = text.split()
    chunks = []

    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        if end >= len(words):
            break

        start += chunk_size - overlap

    return chunks


medical_chunks = []

for page_item in medical_pages:
    page_chunks = chunk_text(
        page_item["text"],
        chunk_size=180,
        overlap=40
    )

    for chunk_number, chunk in enumerate(page_chunks, start=1):
        medical_chunks.append({
            "source": page_item["source"],
            "page": page_item["page"],
            "chunk_id": chunk_number,
            "text": chunk
        })


print("=== Chunking Summary ===")
print("Total chunks created:", len(medical_chunks))

print("\n=== Sample Chunk ===")
print("Source:", medical_chunks[0]["source"])
print("Page:", medical_chunks[0]["page"])
print("Chunk:", medical_chunks[0]["chunk_id"])
print("\nText:")
print(medical_chunks[0]["text"])

=== Chunking Summary ===
Total chunks created: 14

=== Sample Chunk ===
Source: WHO_Diabetes_Fact_Sheet.pdf
Page: 1
Chunk: 1

Text:
Quick facts Globally, an estimated 346 million people have diabetes. Three out of four people with diabetes live in low- and • middle-income countries. In the South-East Asia (SEA) Region, nearly 71 million were estimated to be living with diabetes in 2010 and an • equal number had impaired glucose tolerance. Nearly 3.4 million people globally and 1 million in SEA Region die from consequences of high blood sugar every • year. Diabetes exacerbates major infectious diseases such as tuberculosis (TB), malaria and HIV/AIDS. People with • diabetes are three times more likely to develop TB when infected and approximately 15% of TB globally is thought to be due to diabetes. What is diabetes? Diabetes is a chronic condition that occurs when blood glucose levels remain above normal limits. This happens if the pancreas does not produce enough insulin (a hormone that

## 8. Creating Embeddings and the FAISS Vector Index

Each medical text chunk is converted into a semantic vector using a Sentence Transformer embedding model. The embeddings are normalized and stored in a FAISS vector index, allowing the RAG system to retrieve chunks that are semantically similar to a user's question.

The embedding model is kept on the CPU so that GPU memory remains available for the Dynamic 4-bit language model.

In [9]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load the embedding model on CPU to preserve GPU VRAM
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu"
)

# Collect chunk texts
chunk_texts = [item["text"] for item in medical_chunks]

# Create normalized semantic embeddings
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

# FAISS requires float32
chunk_embeddings = np.asarray(
    chunk_embeddings,
    dtype="float32"
)

# Build FAISS index using inner product.
# With normalized embeddings, this acts like cosine similarity.
embedding_dimension = chunk_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(chunk_embeddings)

print("✅ Embeddings created successfully")
print("✅ FAISS index created successfully")

print("\n=== Vector Index Summary ===")
print("Embedding model: all-MiniLM-L6-v2")
print("Embedding dimension:", embedding_dimension)
print("Documents indexed:", faiss_index.ntotal)
print("Embedding device: CPU")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings created successfully
✅ FAISS index created successfully

=== Vector Index Summary ===
Embedding model: all-MiniLM-L6-v2
Embedding dimension: 384
Documents indexed: 14
Embedding device: CPU


## 9. Semantic Retrieval Test

A user question is converted into an embedding and compared with the indexed medical-document chunks using FAISS. The most semantically relevant chunks are returned together with their similarity scores, source documents, and page numbers.

In [10]:
# Retrieve the most relevant medical chunks for a user question

def retrieve_chunks(query, top_k=3):
    # Convert the query into a normalized embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    query_embedding = np.asarray(
        query_embedding,
        dtype="float32"
    )

    # Search the FAISS index
    scores, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, index in zip(scores[0], indices[0]):
        item = medical_chunks[index]

        results.append({
            "score": float(score),
            "source": item["source"],
            "page": item["page"],
            "chunk_id": item["chunk_id"],
            "text": item["text"]
        })

    return results


# Test the retrieval system
test_query = "What are the risk factors and prevention measures for hypertension?"

retrieved_results = retrieve_chunks(test_query, top_k=3)

print("=== Retrieval Test ===")
print("Question:", test_query)

for i, result in enumerate(retrieved_results, start=1):
    print("\n" + "=" * 80)
    print(f"Result {i}")
    print(f"Similarity score: {result['score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page']}")
    print(f"Chunk: {result['chunk_id']}")
    print("\nRetrieved text:")
    print(result["text"])

=== Retrieval Test ===
Question: What are the risk factors and prevention measures for hypertension?

Result 1
Similarity score: 0.6771
Source: WHO_Pakistan_Hypertension_Fact_Sheet.pdf
Page: 1
Chunk: 2

Retrieved text:
prevalence of hypertension through reducing modifiable risk factors such as unhealthy diets (excessive salt consumption, high in saturated fat and trans fats, low intake of fruits and vegetables), physical inactivity, consumption of tobacco and alcohol, and being overweight or obese. WHO MPOWER, ACTIVE, SHAKE, REPLACE and HEARTS technical packages** can help in this. • Address hypertension control (WHO HEARTS technical package): • Improve and expand identification and treatment (using evidence based protocols) of people with hypertension • Treatment intensification for patients whose blood pressure isn’t controlled and ensuring access to medicines • Track blood pressure control rates in clinical settings and measure population prevalence. WHO RECOMMENDATIONS FOR HYPERTEN

## 10. Retrieval-Augmented Generation

The retrieval component is now connected to the Unsloth Dynamic 4-bit language model. For each user question, the most relevant WHO document chunks are retrieved from the FAISS index and supplied to the language model as context.

The model is instructed to answer only from the retrieved evidence. This helps produce grounded responses and reduces unsupported generation.

In [11]:
# Connect semantic retrieval with the Dynamic 4-bit LLM

def generate_rag_answer(query, top_k=3, max_new_tokens=250):

    # Retrieve the most relevant medical chunks
    retrieved = retrieve_chunks(query, top_k=top_k)

    # Build grounded context
    context_parts = []

    for i, item in enumerate(retrieved, start=1):
        context_parts.append(
            f"[Source {i}: {item['source']}, Page {item['page']}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_parts)

    # Instruction for grounded generation
    messages = [
        {
            "role": "system",
            "content": (
                "You are a medical information assistant. "
                "Answer the user's question using only the retrieved context provided. "
                "Do not invent information that is not supported by the context. "
                "If the context is insufficient, clearly say that the available "
                "documents do not provide enough information."
            )
        },
        {
            "role": "user",
            "content": (
                f"Retrieved medical context:\n\n{context}\n\n"
                f"Question: {query}\n\n"
                "Provide a concise, evidence-grounded answer."
            )
        }
    ]

    # Apply Llama chat template
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize while leaving room for generated tokens
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1600
    ).to("cuda")

    # Generate answer
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode only newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer, retrieved

In [12]:
# Test the complete RAG pipeline

rag_question = "What are the main risk factors and prevention measures for hypertension?"

rag_answer, rag_sources = generate_rag_answer(
    rag_question,
    top_k=3
)

print("=== Medical RAG Response ===")
print("\nQuestion:")
print(rag_question)

print("\nGrounded Answer:")
print(rag_answer)

print("\n=== Retrieved Sources ===")

for i, source in enumerate(rag_sources, start=1):
    print(
        f"{i}. {source['source']} | "
        f"Page {source['page']} | "
        f"Similarity: {source['score']:.4f}"
    )

Both `max_new_tokens` (=250) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== Medical RAG Response ===

Question:
What are the main risk factors and prevention measures for hypertension?

Grounded Answer:
According to the provided context, the main risk factors for hypertension are:

1. Unhealthy diets (excessive salt consumption, high in saturated fat and trans fats, low intake of fruits and vegetables)
2. Physical inactivity
3. Consumption of tobacco and alcohol
4. Being overweight or obese

Prevention measures mentioned in the context include:

1. WHO MPOWER, ACTIVE, SHAKE, REPLACE, and HEARTS technical packages
2. Improving and expanding identification and treatment of people with hypertension using evidence-based protocols
3. Treatment intensification for patients whose blood pressure isn't controlled
4. Ensuring access to medicines
5. Tracking blood pressure control rates in clinical settings and measuring population prevalence.

These measures aim to address the risk factors mentioned above.

=== Retrieved Sources ===
1. WHO_Pakistan_Hypertension_Fact

## 11. Retrieval Quality Evaluation

Before extending the RAG system, the retrieval component is evaluated using questions from different medical topics. The tests include hypertension, diabetes, and an out-of-domain query to examine whether FAISS retrieves the appropriate source documents and to identify cases where a confidence threshold may be needed.

In [13]:
# Evaluate retrieval quality across multiple medical questions

evaluation_queries = [
    "What are the main risk factors for hypertension?",
    "How can hypertension be prevented?",
    "What blood pressure levels are used when discussing hypertension?",
    "How many people globally are estimated to have diabetes?",
    "What are important facts about diabetes?",
    "What are the symptoms and treatment of asthma?"
]

print("=== Retrieval Quality Evaluation ===")

for query in evaluation_queries:
    results = retrieve_chunks(query, top_k=3)

    print("\n" + "=" * 90)
    print("Query:", query)

    for rank, result in enumerate(results, start=1):
        print(
            f"\nRank {rank} | "
            f"Score: {result['score']:.4f} | "
            f"Source: {result['source']} | "
            f"Page: {result['page']}"
        )

        print("Preview:", result["text"][:220], "...")

=== Retrieval Quality Evaluation ===

Query: What are the main risk factors for hypertension?

Rank 1 | Score: 0.5792 | Source: WHO_Pakistan_Hypertension_Fact_Sheet.pdf | Page: 2
Preview: AGE AND SEX (2013)1,2 TRENDS IN UNCONTROLLED HYPERTENSION PREVALENCE IN ADULTS AGED 18+4 2000 2005 2010 2015 2020 2025 0 10 20 30 40 Males [N in 100 000 (%)] Females [N in 100 000 (%)] 18-69 18-29 30-44 45-59 60-69 18-69 ...

Rank 2 | Score: 0.5652 | Source: WHO_Pakistan_Hypertension_Fact_Sheet.pdf | Page: 1
Preview: prevalence of hypertension through reducing modifiable risk factors such as unhealthy diets (excessive salt consumption, high in saturated fat and trans fats, low intake of fruits and vegetables), physical inactivity, co ...

Rank 3 | Score: 0.5625 | Source: WHO_Pakistan_Hypertension_Fact_Sheet.pdf | Page: 2
Preview: who: • Have SBP of < 140 mmHg (mean of 2nd and 3rd measurements), AND • Have DBP of < 90 mmHg (mean of 2nd and 3rd measurements), AND • Have been told by a doctor or other he

## 12. Evidence Confidence and Out-of-Domain Protection

A retrieval-confidence check is added to reduce unsupported answers. During evaluation, relevant hypertension and diabetes questions produced substantially higher similarity scores than an out-of-domain asthma question. A provisional similarity threshold is therefore used to determine whether the indexed documents contain sufficient evidence before sending context to the language model.

This threshold is treated as empirical and should be recalibrated if the knowledge base is expanded.

In [14]:
# Add a confidence gate to prevent unsupported RAG responses

RETRIEVAL_THRESHOLD = 0.35


def retrieve_with_confidence(query, top_k=3, threshold=RETRIEVAL_THRESHOLD):

    results = retrieve_chunks(query, top_k=top_k)

    if not results:
        return {
            "accepted": False,
            "reason": "No documents were retrieved.",
            "results": []
        }

    top_score = results[0]["score"]

    if top_score < threshold:
        return {
            "accepted": False,
            "reason": (
                f"Insufficient retrieval confidence "
                f"(top similarity = {top_score:.4f}, "
                f"required >= {threshold:.2f})."
            ),
            "results": results
        }

    return {
        "accepted": True,
        "reason": (
            f"Relevant evidence found "
            f"(top similarity = {top_score:.4f})."
        ),
        "results": results
    }

In [15]:
# Test the confidence gate with in-domain and out-of-domain questions

confidence_test_queries = [
    "How can hypertension be prevented?",
    "How many people globally are estimated to have diabetes?",
    "What are the symptoms and treatment of asthma?"
]

for query in confidence_test_queries:

    check = retrieve_with_confidence(query)

    print("\n" + "=" * 90)
    print("Question:", query)
    print("Accepted:", check["accepted"])
    print("Decision:", check["reason"])

    if check["results"]:
        print(
            "Top source:",
            check["results"][0]["source"],
            "| Score:",
            f"{check['results'][0]['score']:.4f}"
        )


Question: How can hypertension be prevented?
Accepted: True
Decision: Relevant evidence found (top similarity = 0.6569).
Top source: WHO_Pakistan_Hypertension_Fact_Sheet.pdf | Score: 0.6569

Question: How many people globally are estimated to have diabetes?
Accepted: True
Decision: Relevant evidence found (top similarity = 0.6931).
Top source: WHO_Diabetes_Fact_Sheet.pdf | Score: 0.6931

Question: What are the symptoms and treatment of asthma?
Accepted: False
Decision: Insufficient retrieval confidence (top similarity = 0.1810, required >= 0.35).
Top source: WHO_Diabetes_Fact_Sheet.pdf | Score: 0.1810


## 13. Guarded RAG Generation

The RAG pipeline is extended with an evidence-confidence gate. Before generating an answer, the system evaluates whether the retrieved document evidence is sufficiently relevant.

If the retrieval confidence is below the selected threshold, the language model is not used to generate a potentially unsupported medical answer. Instead, the system returns a clear message indicating that the current knowledge base does not contain enough evidence.

In [16]:
# Guarded RAG pipeline with retrieval-confidence protection

def generate_guarded_rag_answer(
    query,
    top_k=3,
    threshold=RETRIEVAL_THRESHOLD,
    max_new_tokens=250
):

    # Step 1: Retrieve evidence and evaluate confidence
    retrieval_check = retrieve_with_confidence(
        query,
        top_k=top_k,
        threshold=threshold
    )

    retrieved = retrieval_check["results"]

    # Step 2: Refuse generation when evidence is insufficient
    if not retrieval_check["accepted"]:

        answer = (
            "I do not have enough reliable evidence in the indexed "
            "medical documents to answer this question. "
            "Please consult an appropriate authoritative source or "
            "expand the knowledge base with relevant documents."
        )

        return {
            "question": query,
            "answer": answer,
            "generated": False,
            "decision": retrieval_check["reason"],
            "sources": retrieved
        }

    # Step 3: Build context from retrieved evidence
    context_parts = []

    for i, item in enumerate(retrieved, start=1):
        context_parts.append(
            f"[Source {i}: {item['source']}, Page {item['page']}]\n"
            f"{item['text']}"
        )

    context = "\n\n".join(context_parts)

    # Step 4: Create grounded chat prompt
    messages = [
        {
            "role": "system",
            "content": (
                "You are a medical information assistant. "
                "Answer only from the retrieved evidence provided. "
                "Do not add unsupported medical claims. "
                "If information is incomplete, clearly state the limitation. "
                "This system provides educational information and does not "
                "replace professional medical advice."
            )
        },
        {
            "role": "user",
            "content": (
                f"Retrieved evidence:\n\n{context}\n\n"
                f"Question: {query}\n\n"
                "Provide a concise answer grounded in the evidence."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1600
    ).to("cuda")

    # Step 5: Generate grounded response
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return {
        "question": query,
        "answer": answer,
        "generated": True,
        "decision": retrieval_check["reason"],
        "sources": retrieved
    }

In [17]:
# Test guarded RAG with one supported and one unsupported question

guarded_test_queries = [
    "How can hypertension be prevented?",
    "What are the symptoms and treatment of asthma?"
]

for query in guarded_test_queries:

    result = generate_guarded_rag_answer(query)

    print("\n" + "=" * 90)
    print("Question:", result["question"])
    print("LLM generation used:", result["generated"])
    print("Retrieval decision:", result["decision"])

    print("\nAnswer:")
    print(result["answer"])

    if result["sources"]:
        print("\nRetrieved evidence:")

        for i, source in enumerate(result["sources"], start=1):
            print(
                f"{i}. {source['source']} | "
                f"Page {source['page']} | "
                f"Similarity {source['score']:.4f}"
            )

Both `max_new_tokens` (=250) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Question: How can hypertension be prevented?
LLM generation used: True
Retrieval decision: Relevant evidence found (top similarity = 0.6569).

Answer:
According to the World Health Organization (WHO) fact sheet, hypertension can be prevented by reducing modifiable risk factors such as:

1. Unhealthy diets (excessive salt consumption, high in saturated fat and trans fats, low intake of fruits and vegetables)
2. Physical inactivity
3. Consumption of tobacco and alcohol
4. Being overweight or obese

Additionally, WHO recommends using evidence-based protocols for identification and treatment of people with hypertension, intensifying treatment for those whose blood pressure isn't controlled, and tracking blood pressure control rates in clinical settings.

Specifically, the WHO recommends:

* Having a systolic blood pressure (SBP) of < 140 mmHg and a diastolic blood pressure (DBP) of < 90 mmHg
* Being told by a doctor or health worker that they have raised blood pressure or hypertension
* H

## 14. Retrieval Reranking for Higher-Quality Evidence

FAISS efficiently retrieves semantically similar candidate chunks, but semantic similarity alone does not guarantee that every retrieved passage directly answers the user's question.

A Cross-Encoder reranker is therefore added after FAISS retrieval. The reranker evaluates each question–passage pair jointly and reorders the candidates so that the most directly relevant evidence is passed to the language model. This two-stage retrieval strategy improves evidence quality and reduces the risk of generating answers from loosely related context.

In [18]:
from sentence_transformers import CrossEncoder

# Load a lightweight reranking model on CPU
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu"
)

print("✅ Cross-Encoder reranker loaded successfully")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Cross-Encoder reranker loaded successfully


In [19]:
def retrieve_and_rerank(query, candidate_k=6, final_k=3):

    # Stage 1: FAISS candidate retrieval
    candidates = retrieve_chunks(
        query,
        top_k=min(candidate_k, len(medical_chunks))
    )

    # Create question-passage pairs
    pairs = [
        [query, candidate["text"]]
        for candidate in candidates
    ]

    # Stage 2: Cross-Encoder reranking
    rerank_scores = reranker.predict(pairs)

    for candidate, rerank_score in zip(candidates, rerank_scores):
        candidate["rerank_score"] = float(rerank_score)

    # Highest reranking score first
    reranked = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    return reranked[:final_k]

In [20]:
query = "How can hypertension be prevented?"

reranked_results = retrieve_and_rerank(
    query,
    candidate_k=6,
    final_k=3
)

print("=== Reranked Retrieval Results ===")
print("Question:", query)

for rank, result in enumerate(reranked_results, start=1):

    print("\n" + "=" * 80)
    print(f"Rank {rank}")
    print(f"FAISS similarity: {result['score']:.4f}")
    print(f"Reranker score: {result['rerank_score']:.4f}")
    print(f"Source: {result['source']}")
    print(f"Page: {result['page']}")

    print("\nEvidence preview:")
    print(result["text"][:500])

=== Reranked Retrieval Results ===
Question: How can hypertension be prevented?

Rank 1
FAISS similarity: 0.6569
Reranker score: -1.6170
Source: WHO_Pakistan_Hypertension_Fact_Sheet.pdf
Page: 1

Evidence preview:
prevalence of hypertension through reducing modifiable risk factors such as unhealthy diets (excessive salt consumption, high in saturated fat and trans fats, low intake of fruits and vegetables), physical inactivity, consumption of tobacco and alcohol, and being overweight or obese. WHO MPOWER, ACTIVE, SHAKE, REPLACE and HEARTS technical packages** can help in this. • Address hypertension control (WHO HEARTS technical package): • Improve and expand identification and treatment (using evidence b

Rank 2
FAISS similarity: 0.4310
Reranker score: -4.9985
Source: WHO_Diabetes_Fact_Sheet.pdf
Page: 2

Evidence preview:
How is diabetes diagnosed? Early diagnosis can be accomplished through blood testing, such as fasting or random blood glucose test, oral glucose tolerance test, or gl

## 15. Dynamic Evidence Selection

Reranking improves passage relevance, but blindly passing a fixed number of top-ranked chunks can still introduce loosely related evidence.

The pipeline therefore applies two levels of filtering:

1. A minimum FAISS similarity requirement removes weak semantic matches.
2. Cross-Encoder scores are converted into relative confidence values, allowing only strongly supported reranked passages to enter the final LLM context.

This produces a smaller, cleaner evidence set and reduces irrelevant context.

In [21]:
def retrieve_rerank_select(
    query,
    candidate_k=8,
    final_k=3,
    faiss_threshold=0.35,
    rerank_probability_threshold=0.05
):

    # Stage 1: FAISS retrieval
    candidates = retrieve_chunks(
        query,
        top_k=min(candidate_k, len(medical_chunks))
    )

    # Remove weak semantic matches
    candidates = [
        item for item in candidates
        if item["score"] >= faiss_threshold
    ]

    if not candidates:
        return []

    # Stage 2: Cross-Encoder reranking
    pairs = [
        [query, item["text"]]
        for item in candidates
    ]

    rerank_scores = np.asarray(
        reranker.predict(pairs),
        dtype=np.float32
    )

    # Convert raw reranker logits into relative probabilities
    shifted_scores = rerank_scores - np.max(rerank_scores)
    probabilities = np.exp(shifted_scores)
    probabilities = probabilities / probabilities.sum()

    for item, rerank_score, probability in zip(
        candidates,
        rerank_scores,
        probabilities
    ):
        item["rerank_score"] = float(rerank_score)
        item["rerank_probability"] = float(probability)

    # Sort by reranker score
    candidates = sorted(
        candidates,
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    # Keep only strongly supported evidence
    selected = [
        item for item in candidates
        if item["rerank_probability"] >= rerank_probability_threshold
    ]

    # Always preserve the strongest passage if evidence exists
    if not selected:
        selected = [candidates[0]]

    return selected[:final_k]

In [22]:
selection_test_queries = [
    "How can hypertension be prevented?",
    "How many people globally are estimated to have diabetes?",
    "What are the symptoms and treatment of asthma?"
]

for query in selection_test_queries:

    selected = retrieve_rerank_select(query)

    print("\n" + "=" * 90)
    print("Question:", query)
    print("Evidence selected:", len(selected))

    if not selected:
        print("No sufficiently relevant evidence found.")
        continue

    for rank, item in enumerate(selected, start=1):
        print(
            f"\nRank {rank} | "
            f"FAISS: {item['score']:.4f} | "
            f"Reranker: {item['rerank_score']:.4f} | "
            f"Relative confidence: "
            f"{item['rerank_probability']:.4f}"
        )

        print(
            f"Source: {item['source']} | "
            f"Page: {item['page']}"
        )

        print("Preview:", item["text"][:300], "...")


Question: How can hypertension be prevented?
Evidence selected: 1

Rank 1 | FAISS: 0.6569 | Reranker: -1.6170 | Relative confidence: 0.9605
Source: WHO_Pakistan_Hypertension_Fact_Sheet.pdf | Page: 1
Preview: prevalence of hypertension through reducing modifiable risk factors such as unhealthy diets (excessive salt consumption, high in saturated fat and trans fats, low intake of fruits and vegetables), physical inactivity, consumption of tobacco and alcohol, and being overweight or obese. WHO MPOWER, ACT ...

Question: How many people globally are estimated to have diabetes?
Evidence selected: 1

Rank 1 | FAISS: 0.6931 | Reranker: 10.0953 | Relative confidence: 1.0000
Source: WHO_Diabetes_Fact_Sheet.pdf | Page: 1
Preview: Quick facts Globally, an estimated 346 million people have diabetes. Three out of four people with diabetes live in low- and • middle-income countries. In the South-East Asia (SEA) Region, nearly 71 million were estimated to be living with diabetes in 2010 and an • eq

## 16. Final Reranked and Guarded RAG Pipeline

The final RAG pipeline combines semantic retrieval, evidence filtering, Cross-Encoder reranking, and Dynamic 4-bit language-model inference.

Only sufficiently relevant evidence is supplied to the language model. If no reliable evidence remains after retrieval and reranking, generation is skipped and the system returns a controlled insufficient-evidence response. This design improves grounding and reduces unsupported medical answers.

In [23]:
import time

def final_medical_rag(
    query,
    candidate_k=8,
    final_k=3,
    faiss_threshold=0.35,
    rerank_probability_threshold=0.05,
    max_new_tokens=220
):

    start_time = time.time()

    # -------------------------------------------------
    # 1. Retrieve, rerank, and select reliable evidence
    # -------------------------------------------------
    selected_evidence = retrieve_rerank_select(
        query=query,
        candidate_k=candidate_k,
        final_k=final_k,
        faiss_threshold=faiss_threshold,
        rerank_probability_threshold=rerank_probability_threshold
    )

    # -------------------------------------------------
    # 2. Stop generation if evidence is insufficient
    # -------------------------------------------------
    if not selected_evidence:

        elapsed_time = time.time() - start_time

        return {
            "question": query,
            "answer": (
                "The indexed medical knowledge base does not contain "
                "sufficient reliable evidence to answer this question. "
                "Please consult an appropriate authoritative medical source."
            ),
            "generated": False,
            "sources": [],
            "response_time": elapsed_time
        }

    # -------------------------------------------------
    # 3. Build evidence context
    # -------------------------------------------------
    context_parts = []

    for i, item in enumerate(selected_evidence, start=1):

        context_parts.append(
            f"[Evidence {i}]\n"
            f"Source: {item['source']}\n"
            f"Page: {item['page']}\n"
            f"Content: {item['text']}"
        )

    context = "\n\n".join(context_parts)

    # -------------------------------------------------
    # 4. Create grounded prompt
    # -------------------------------------------------
    messages = [
        {
            "role": "system",
            "content": (
                "You are an evidence-grounded medical information assistant. "
                "Use only the supplied evidence to answer the question. "
                "Do not add medical claims that are unsupported by the evidence. "
                "If the evidence only partially answers the question, clearly "
                "state that limitation. Keep the answer concise and factual. "
                "This information is educational and does not replace "
                "professional medical advice."
            )
        },
        {
            "role": "user",
            "content": (
                f"Evidence:\n\n{context}\n\n"
                f"Question: {query}\n\n"
                "Answer using only the evidence above."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1600
    ).to("cuda")

    # -------------------------------------------------
    # 5. Dynamic 4-bit LLM generation
    # -------------------------------------------------
    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    elapsed_time = time.time() - start_time

    return {
        "question": query,
        "answer": answer,
        "generated": True,
        "sources": selected_evidence,
        "response_time": elapsed_time
    }

In [24]:
final_test_queries = [
    "How can hypertension be prevented?",
    "How many people globally are estimated to have diabetes?",
    "What are the symptoms and treatment of asthma?"
]

for query in final_test_queries:

    result = final_medical_rag(query)

    print("\n" + "=" * 95)
    print("QUESTION:")
    print(result["question"])

    print("\nLLM GENERATION USED:")
    print(result["generated"])

    print("\nANSWER:")
    print(result["answer"])

    print(
        f"\nResponse time: "
        f"{result['response_time']:.2f} seconds"
    )

    if result["sources"]:

        print("\nEVIDENCE SOURCES:")

        for i, source in enumerate(
            result["sources"],
            start=1
        ):

            print(
                f"{i}. {source['source']} | "
                f"Page {source['page']} | "
                f"FAISS {source['score']:.4f} | "
                f"Rerank confidence "
                f"{source['rerank_probability']:.4f}"
            )

Both `max_new_tokens` (=220) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
How can hypertension be prevented?

LLM GENERATION USED:
True

ANSWER:
According to the evidence, hypertension can be prevented by reducing modifiable risk factors such as:

1. Unhealthy diets (excessive salt consumption, high in saturated fat and trans fats, low intake of fruits and vegetables)
2. Physical inactivity
3. Consumption of tobacco and alcohol
4. Being overweight or obese.

Additionally, the WHO recommends using evidence-based protocols for identification and treatment of people with hypertension, as well as intensifying treatment for those whose blood pressure isn't controlled, to help prevent hypertension.

Response time: 5.70 seconds

EVIDENCE SOURCES:
1. WHO_Pakistan_Hypertension_Fact_Sheet.pdf | Page 1 | FAISS 0.6569 | Rerank confidence 0.9605


Both `max_new_tokens` (=220) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
How many people globally are estimated to have diabetes?

LLM GENERATION USED:
True

ANSWER:
According to the evidence, globally, an estimated 346 million people have diabetes.

Response time: 1.50 seconds

EVIDENCE SOURCES:
1. WHO_Diabetes_Fact_Sheet.pdf | Page 1 | FAISS 0.6931 | Rerank confidence 1.0000

QUESTION:
What are the symptoms and treatment of asthma?

LLM GENERATION USED:
False

ANSWER:
The indexed medical knowledge base does not contain sufficient reliable evidence to answer this question. Please consult an appropriate authoritative medical source.

Response time: 0.02 seconds


## 17. Clean Generation Configuration

The model generation configuration is adjusted so that response length is controlled only through `max_new_tokens`. This removes conflicting generation-length settings and keeps the final inference pipeline clean and reproducible.

In [25]:
# Remove the inherited max_length setting to avoid generation warnings

model.generation_config.max_length = None

print("✅ Generation configuration cleaned")
print("max_length:", model.generation_config.max_length)

✅ Generation configuration cleaned
max_length: None


In [26]:
clean_test = final_medical_rag(
    "How many people globally are estimated to have diabetes?"
)

print("QUESTION:")
print(clean_test["question"])

print("\nANSWER:")
print(clean_test["answer"])

print("\nLLM generation used:")
print(clean_test["generated"])

print(
    "\nResponse time:",
    f"{clean_test['response_time']:.2f} seconds"
)

QUESTION:
How many people globally are estimated to have diabetes?

ANSWER:
According to the evidence, globally, an estimated 346 million people have diabetes.

LLM generation used:
True

Response time: 1.54 seconds


## 18. End-to-End System Evaluation

The final RAG pipeline is evaluated using a small structured test set containing both supported and unsupported medical questions.

The evaluation focuses on:

- correct routing to the expected medical document,
- correct generation or refusal behavior,
- response latency,
- evidence selection behavior.

These metrics evaluate retrieval and safety behavior. They should not be interpreted as a clinical-accuracy benchmark.

In [27]:
import pandas as pd

evaluation_cases = [
    {
        "question": "How can hypertension be prevented?",
        "expected_source": "WHO_Pakistan_Hypertension_Fact_Sheet.pdf",
        "should_generate": True
    },
    {
        "question": "What are the main risk factors for hypertension?",
        "expected_source": "WHO_Pakistan_Hypertension_Fact_Sheet.pdf",
        "should_generate": True
    },
    {
        "question": "What blood pressure levels are discussed in the hypertension document?",
        "expected_source": "WHO_Pakistan_Hypertension_Fact_Sheet.pdf",
        "should_generate": True
    },
    {
        "question": "What does the WHO document say about hypertension control?",
        "expected_source": "WHO_Pakistan_Hypertension_Fact_Sheet.pdf",
        "should_generate": True
    },
    {
        "question": "How many people globally are estimated to have diabetes?",
        "expected_source": "WHO_Diabetes_Fact_Sheet.pdf",
        "should_generate": True
    },
    {
        "question": "What are important facts about diabetes?",
        "expected_source": "WHO_Diabetes_Fact_Sheet.pdf",
        "should_generate": True
    },
    {
        "question": "How is diabetes diagnosed?",
        "expected_source": "WHO_Diabetes_Fact_Sheet.pdf",
        "should_generate": True
    },
    {
        "question": "What information does the document provide about diabetes symptoms?",
        "expected_source": "WHO_Diabetes_Fact_Sheet.pdf",
        "should_generate": True
    },
    {
        "question": "What are the symptoms and treatment of asthma?",
        "expected_source": None,
        "should_generate": False
    },
    {
        "question": "What are the causes and treatment of migraine?",
        "expected_source": None,
        "should_generate": False
    }
]

In [28]:
evaluation_results = []

for case in evaluation_cases:

    result = final_medical_rag(case["question"])

    top_source = (
        result["sources"][0]["source"]
        if result["sources"]
        else None
    )

    generation_behavior_correct = (
        result["generated"] == case["should_generate"]
    )

    if case["expected_source"] is None:
        source_correct = top_source is None
    else:
        source_correct = top_source == case["expected_source"]

    evaluation_results.append({
        "Question": case["question"],
        "Expected Source": case["expected_source"] or "None",
        "Top Retrieved Source": top_source or "None",
        "Should Generate": case["should_generate"],
        "Generated": result["generated"],
        "Source Correct": source_correct,
        "Behavior Correct": generation_behavior_correct,
        "Response Time (s)": round(result["response_time"], 2),
        "Answer Preview": result["answer"][:120]
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,Question,Expected Source,Top Retrieved Source,Should Generate,Generated,Source Correct,Behavior Correct,Response Time (s),Answer Preview
0,How can hypertension be prevented?,WHO_Pakistan_Hypertension_Fact_Sheet.pdf,WHO_Pakistan_Hypertension_Fact_Sheet.pdf,True,True,True,True,6.64,"According to the evidence, hypertension can be..."
1,What are the main risk factors for hypertension?,WHO_Pakistan_Hypertension_Fact_Sheet.pdf,WHO_Pakistan_Hypertension_Fact_Sheet.pdf,True,True,True,True,4.34,"According to the evidence, the main risk facto..."
2,What blood pressure levels are discussed in th...,WHO_Pakistan_Hypertension_Fact_Sheet.pdf,WHO_Pakistan_Hypertension_Fact_Sheet.pdf,True,True,True,True,5.24,"According to the provided evidence, the blood ..."
3,What does the WHO document say about hypertens...,WHO_Pakistan_Hypertension_Fact_Sheet.pdf,WHO_Pakistan_Hypertension_Fact_Sheet.pdf,True,True,True,True,9.46,"According to the WHO document, the document pr..."
4,How many people globally are estimated to have...,WHO_Diabetes_Fact_Sheet.pdf,WHO_Diabetes_Fact_Sheet.pdf,True,True,True,True,1.59,"According to the evidence, globally, an estima..."
5,What are important facts about diabetes?,WHO_Diabetes_Fact_Sheet.pdf,WHO_Diabetes_Fact_Sheet.pdf,True,True,True,True,10.34,Here are the important facts about diabetes ba...
6,How is diabetes diagnosed?,WHO_Diabetes_Fact_Sheet.pdf,WHO_Diabetes_Fact_Sheet.pdf,True,True,True,True,2.53,Diabetes can be diagnosed through early diagno...
7,What information does the document provide abo...,WHO_Diabetes_Fact_Sheet.pdf,WHO_Diabetes_Fact_Sheet.pdf,True,True,True,True,8.00,"According to the provided evidence, the docume..."
8,What are the symptoms and treatment of asthma?,None,None,False,False,True,True,0.02,The indexed medical knowledge base does not co...
9,What are the causes and treatment of migraine?,None,None,False,False,True,True,0.02,The indexed medical knowledge base does not co...


In [29]:
source_accuracy = evaluation_df["Source Correct"].mean() * 100
behavior_accuracy = evaluation_df["Behavior Correct"].mean() * 100
average_latency = evaluation_df["Response Time (s)"].mean()

supported_df = evaluation_df[
    evaluation_df["Should Generate"] == True
]

unsupported_df = evaluation_df[
    evaluation_df["Should Generate"] == False
]

supported_generation_rate = (
    supported_df["Generated"].mean() * 100
)

refusal_accuracy = (
    (~unsupported_df["Generated"]).mean() * 100
)

print("=== Final Evaluation Summary ===")

print(f"Total evaluation questions: {len(evaluation_df)}")
print(f"Source-routing accuracy: {source_accuracy:.1f}%")
print(f"Generation/refusal behavior accuracy: {behavior_accuracy:.1f}%")
print(f"Supported-query generation rate: {supported_generation_rate:.1f}%")
print(f"Unsupported-query refusal accuracy: {refusal_accuracy:.1f}%")
print(f"Average response time: {average_latency:.2f} seconds")

=== Final Evaluation Summary ===
Total evaluation questions: 10
Source-routing accuracy: 100.0%
Generation/refusal behavior accuracy: 100.0%
Supported-query generation rate: 100.0%
Unsupported-query refusal accuracy: 100.0%
Average response time: 4.82 seconds


## 19. Interactive Medical RAG Demo

A simple interactive interface is added to demonstrate the final RAG system. Users can enter a medical-information question and receive an evidence-grounded response together with the retrieved source document, page number, retrieval confidence, and response time.

The interface is intended for educational demonstration only and does not provide medical diagnosis or clinical advice.

In [30]:
!pip install -q gradio

In [31]:
import gradio as gr


def medical_rag_demo(question):

    if not question or not question.strip():
        return (
            "Please enter a question.",
            "No evidence retrieved.",
            ""
        )

    result = final_medical_rag(question.strip())

    # ------------------------------
    # Format answer
    # ------------------------------
    answer_text = result["answer"]

    if result["generated"]:
        status = "✅ Evidence-supported generation"
    else:
        status = "⚠️ Generation skipped due to insufficient evidence"

    answer_output = (
        f"{status}\n\n"
        f"{answer_text}\n\n"
        "Educational use only — not a substitute for professional medical advice."
    )

    # ------------------------------
    # Format evidence
    # ------------------------------
    if result["sources"]:

        evidence_lines = []

        for i, source in enumerate(result["sources"], start=1):

            evidence_lines.append(
                f"Evidence {i}\n"
                f"Source: {source['source']}\n"
                f"Page: {source['page']}\n"
                f"FAISS similarity: {source['score']:.4f}\n"
                f"Rerank confidence: "
                f"{source['rerank_probability']:.4f}"
            )

        evidence_output = "\n\n".join(evidence_lines)

    else:
        evidence_output = (
            "No sufficiently relevant evidence was found "
            "in the indexed knowledge base."
        )

    # ------------------------------
    # Performance
    # ------------------------------
    performance_output = (
        f"Response time: {result['response_time']:.2f} seconds\n"
        f"LLM generation used: {result['generated']}"
    )

    return (
        answer_output,
        evidence_output,
        performance_output
    )

In [32]:
demo = gr.Interface(
    fn=medical_rag_demo,

    inputs=gr.Textbox(
        lines=2,
        placeholder="Example: How can hypertension be prevented?",
        label="Medical Information Question"
    ),

    outputs=[
        gr.Textbox(
            label="Evidence-Grounded Answer",
            lines=10
        ),

        gr.Textbox(
            label="Retrieved Evidence",
            lines=8
        ),

        gr.Textbox(
            label="System Performance",
            lines=3
        )
    ],

    title="Medical Knowledge RAG Assistant",

    description=(
        "A retrieval-augmented medical information assistant using "
        "WHO documents, FAISS semantic retrieval, Cross-Encoder "
        "reranking, confidence-based evidence selection, and an "
        "Unsloth Dynamic 4-bit Llama 3.2 model."
    ),

    examples=[
        ["How can hypertension be prevented?"],
        ["How many people globally are estimated to have diabetes?"],
        ["What are the symptoms and treatment of asthma?"]
    ]
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://952656c3b40e581e70.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 20. Project Summary, Limitations, and Future Work

### Project Summary

This project developed an evidence-grounded medical Retrieval-Augmented Generation (RAG) system using an Unsloth Dynamic 4-bit Llama 3.2 model.

The final pipeline combines:

- WHO medical documents as the knowledge source
- PDF text extraction and metadata preservation
- overlapping document chunking
- Sentence Transformer embeddings
- FAISS semantic retrieval
- similarity-based evidence filtering
- Cross-Encoder reranking
- confidence-based evidence selection
- guarded LLM generation
- source and page tracking
- GPU-efficient Dynamic 4-bit inference
- an interactive Gradio interface

On the designed 10-question evaluation set, the system achieved:

- **100% source-routing accuracy**
- **100% generation/refusal behavior accuracy**
- **100% supported-query generation rate**
- **100% unsupported-query refusal accuracy**
- **4.82 seconds average response time**

These results describe performance only on the designed evaluation set and should not be interpreted as clinical accuracy.

### Limitations

The current knowledge base is intentionally small and contains WHO documents focused mainly on hypertension and diabetes. Therefore, the system cannot reliably answer questions outside these indexed topics.

Similarity thresholds and reranking settings were selected empirically for this prototype and may require recalibration when the document collection is expanded.

The system is designed for educational medical information retrieval and has not undergone clinical validation.

### Safety Considerations

The assistant generates answers only when sufficiently relevant evidence is retrieved. When reliable evidence is unavailable, generation is skipped rather than allowing the language model to produce an unsupported medical response.

This system is for educational purposes only and is not a substitute for professional medical advice, diagnosis, or treatment.

### Future Work

Future development could include:

- expanding the knowledge base with additional authoritative medical sources
- evaluating the system on a larger benchmark
- adding citation links to original source documents
- improving retrieval with hybrid semantic and keyword search
- deploying the application as a persistent web service
- adding conversation history while preserving evidence grounding